# Week 5: Advanced Ensemble Algorithms
## Notebook 08: LightGBM Core Mechanics: GOSS & Histogram Binning

### 1. Conceptual Intuition & Theoretical Background
Traditional gradient boosting algorithms (like standard XGBoost exact greedy) iterate over all data points and all continuous features to find optimal split points, taking $O(\#\text{data} \times \#\text{features})$.

**LightGBM** introduces two foundational innovations:
1. **GOSS (Gradient-based One-Side Sampling):** Data points with small gradients have already been well-trained. GOSS keeps all instances with large gradients (top $a \%$) and randomly samples a fraction $b \%$ from instances with small gradients. To prevent altering the data distribution, GOSS scales small-gradient samples by a weight multiplier:
$$\text{Weight Multiplier} = \frac{1 - a}{b}$$

$$\tilde{V}_j(d) = \frac{1}{n} \left( \frac{(\sum_{A} g_i + \frac{1-a}{b} \sum_{B} g_i)^2}{n_{l}(d)} + \frac{(\sum_{A} h_i + \frac{1-a}{b} \sum_{B} h_i)^2}{n_{r}(d)} \right)$$

2. **Histogram Binning:** Continuous numerical features are quantized into discrete bins (e.g. 256 bins), reducing memory consumption by $8\times$ and allowing $O(\#\text{bins})$ split search.

---

### 2. Scratch Python Implementation: GOSS Sampling Algorithm

In [ ]:
import numpy as np

def goss_sampling(gradients, hessians, top_rate=0.2, other_rate=0.1):
    """
    Simulates LightGBM Gradient-based One-Side Sampling (GOSS).
    """
    n_samples = len(gradients)
    abs_gradients = np.abs(gradients)
    
    # Sort indices descending by gradient magnitude
    sorted_idx = np.argsort(-abs_gradients)
    
    # Select top 'a' proportion with large gradients
    top_k = int(top_rate * n_samples)
    top_indices = sorted_idx[:top_k]
    
    # Randomly sample 'b' proportion from remaining small gradients
    remaining_indices = sorted_idx[top_k:]
    sample_k = int(other_rate * n_samples)
    sampled_remaining = np.random.choice(remaining_indices, size=sample_k, replace=False)
    
    # Compute compensation weight multiplier for downsampled subset
    weight_multiplier = (1.0 - top_rate) / other_rate
    
    # Combine selected indices
    selected_indices = np.concatenate([top_indices, sampled_remaining])
    
    # Create sample weights vector
    sample_weights = np.ones(len(selected_indices))
    sample_weights[top_k:] = weight_multiplier  # Amplify small gradients
    
    return selected_indices, sample_weights

# Benchmark GOSS simulation on synthetic gradients
np.random.seed(42)
n_instances = 10000
# Exponential distribution: few large errors, many small residuals
mock_gradients = np.random.exponential(scale=1.0, size=n_instances) - 0.5
mock_hessians = np.ones(n_instances)

selected_idx, weights = goss_sampling(mock_gradients, mock_hessians, top_rate=0.20, other_rate=0.10)

# Verify unbiased gradient sum estimation
true_gradient_sum = np.sum(mock_gradients)
selected_gradients = mock_gradients[selected_idx]
estimated_gradient_sum = np.sum(selected_gradients * weights)

print(f"Total Dataset Samples:          {n_instances}")
print(f"Retained GOSS Samples:          {len(selected_idx)} (Reduced to {len(selected_idx)/n_instances*100:.1f}%)")
print(f"True Total Gradient Sum:        {true_gradient_sum:.2f}")
print(f"GOSS Estimated Gradient Sum:    {estimated_gradient_sum:.2f}")
print(f"Estimation Error:               {abs(true_gradient_sum - estimated_gradient_sum)/abs(true_gradient_sum)*100:.2f}%")

### 3. Step-by-Step Code Walkthrough
1. **Top Error Prioritization**: High gradient instances indicate hard-to-predict examples and are fully retained.
2. **Variance Compensation**: Scaling small-gradient examples by $\frac{1-a}{b}$ guarantees the expected gradient sum is an unbiased estimator.
3. **Speedup**: Reduces gradient calculation costs by up to $70\%$ with negligible loss in accuracy.

---

### 4. Industry Applications & Trade-offs
- **High-Dimensional Tabular Kaggle Competitions:** LightGBM trains up to $20\times$ faster than standard tree implementations.
- **Trade-off:** Leaf-wise (best-first) tree growth can cause overfitting on tiny datasets if `max_depth` or `min_child_samples` is unconstrained.

---

### 5. Top Interview Questions & Answers
**Q1: What is the theoretical reason why GOSS does not introduce bias into gradient boosting trees?**
> **Answer:** Because GOSS applies the correction weight multiplier $\frac{1-a}{b}$ to the randomly sampled subset with smaller gradients, the expectation of the compensated gradient sum equals the true population gradient sum.

**Q2: How does Histogram-based feature binning speed up split evaluation?**
> **Answer:** Rather than sorting continuous feature values in $O(N \log N)$, values are pre-binned into $K$ discrete bins (e.g., $K=256$). Constructing the histogram takes $O(N)$, and evaluating splits takes only $O(K)$.